In [22]:
from pathlib import Path
import pandas as pd
from sqlalchemy import and_, case, distinct, func, literal, or_

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,
    TLS202_APPLN_TITLE,
    TLS203_APPLN_ABSTR,
    TLS224_APPLN_CPC,
)

ENV = "TEST"  # Lavoriamo in test per essere veloci
OUTPUT_DIR = Path("outputs/step0")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

patstat = PatstatClient(env=ENV)
db = patstat.orm()

In [23]:
SECTOR_NAME = "Quantum Computing"
SECTOR_CPC_PREFIXES = (
  "G06N10/00",  # Computing arrangements based on quantum models (Core)
    "G06N10/20",  # Quantum computation architectures e.g. qubit layouts
    "G06N10/40",  # Quantum processor physical realizations
    "G06N10/60",  # Quantum algorithms
    "B82Y 10/00", # Nanotechnology for information processing (se accompagnata da keyword)   
)

SECTOR_KEYWORDS = (
   "quantum computer",
    "quantum computing",
    "qubit",
    "quantum processor",
    "quantum algorithm",
    "superconducting qubit",
    "quantum error correction",
    "quantum annealing",
    "post-quantum cryptography",
    "josephson junction",
)

START_YEAR = "2010-01-01"
END_YEAR = "2023-12-31"

def symbol_prefix_condition(symbol_column, codes):
    symbol = func.replace(symbol_column, " ", "")
    return or_(*(symbol.like(code.replace(" ", "") + "%") for code in codes))

def keyword_condition(terms):
    title = func.lower(TLS202_APPLN_TITLE.appln_title)
    abstract = func.lower(TLS203_APPLN_ABSTR.appln_abstract)
    return or_(*(col.like(f"%{t.lower()}%") for t in terms for col in (title, abstract)))

def sector_family_ids():
    family_id = TLS201_APPLN.docdb_family_id.label("docdb_family_id")
    
    # Perimetro CPC filtrato per intervallo temporale
    by_cpc = (
        db.query(family_id)
        .join(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)
        .filter(symbol_prefix_condition(TLS224_APPLN_CPC.cpc_class_symbol, SECTOR_CPC_PREFIXES))
        .filter(TLS201_APPLN.appln_filing_date >= START_YEAR)
        .filter(TLS201_APPLN.appln_filing_date <= END_YEAR)
    )
    
    if not SECTOR_KEYWORDS:
        return by_cpc.distinct()
        
    # Perimetro Keywords filtrato per intervallo temporale
    by_keywords = (
        db.query(family_id)
        .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
        .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
        .filter(keyword_condition(SECTOR_KEYWORDS))
        .filter(TLS201_APPLN.appln_filing_date >= START_YEAR)
        .filter(TLS201_APPLN.appln_filing_date <= END_YEAR)
    )
    
    return by_cpc.union(by_keywords)

sector_ids = sector_family_ids()

In [24]:
# 1. Campione di famiglie POSITIVE che hanno almeno un titolo o abstract
pos_pool = (
    db.query(TLS201_APPLN.docdb_family_id)
    .filter(TLS201_APPLN.docdb_family_id.in_(sector_ids.subquery().select()))
    .join(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id, isouter=True)
    .join(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id, isouter=True)
    .filter(or_(TLS202_APPLN_TITLE.appln_title.isnot(None), TLS203_APPLN_ABSTR.appln_abstract.isnot(None)))
    .distinct()
    .limit(50)
)
df_pos = patstat.df(pos_pool)
df_pos["query_label"] = 1

# 2. Campione di famiglie NEGATIVE che hanno almeno un titolo o abstract
neg_pool = (
    db.query(TLS201_APPLN.docdb_family_id)
    .filter(~TLS201_APPLN.docdb_family_id.in_(sector_ids.subquery().select()))
    .join(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id, isouter=True)
    .join(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id, isouter=True)
    .filter(or_(TLS202_APPLN_TITLE.appln_title.isnot(None), TLS203_APPLN_ABSTR.appln_abstract.isnot(None)))
    .distinct()
    .limit(50)
)
df_neg = patstat.df(neg_pool)
df_neg["query_label"] = 0

# Uniamo e creiamo il campione totale
df_combined = pd.concat([df_pos, df_neg]).reset_index(drop=True)
sample_size = min(100, len(df_combined))
df_validation = df_combined.sample(n=sample_size, random_state=42, replace=False).reset_index(drop=True)

family_ids_list = df_validation["docdb_family_id"].tolist()

# 3. Recuperiamo titolo e abstract aggregati per il campione
text_query = (
    db.query(
        TLS201_APPLN.docdb_family_id,
        func.max(TLS202_APPLN_TITLE.appln_title).label("appln_title"),
        func.max(TLS203_APPLN_ABSTR.appln_abstract).label("appln_abstract")
    )
    .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
    .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
    .filter(TLS201_APPLN.docdb_family_id.in_(family_ids_list))
    .group_by(TLS201_APPLN.docdb_family_id)
)
df_texts = patstat.df(text_query)

df_validation = df_validation.merge(df_texts, on="docdb_family_id", how="left")
df_validation["manual_label"] = ""

validation_csv_path = OUTPUT_DIR / "validation_sample_100.csv"
df_validation.to_csv(validation_csv_path, index=False)

print(f"Campione generato con successo ({len(df_validation)} righe con testo): {validation_csv_path}")
display(df_validation.head(5))

Campione generato con successo (83 righe con testo): outputs/step0/validation_sample_100.csv


,docdb_family_id,query_label,appln_title,appln_abstract,manual_label
0,41136066,1,COATINGS CONTAINING FUNCTIONALIZED GRAPHENE SH...,Coatings comprising functionalized graphene sh...,
1,60242617,1,確率制御問題に対する方策を改善する方法及びシステム,A method and system for improving a stochastic...,
2,73796908,1,Power transmission line icing thickness predic...,The invention discloses a power transmission l...,
3,83286432,1,Systems and methods for post-quantum cryptogra...,"Systems, apparatuses, methods, and computer pr...",
4,44646354,1,MASTER MOLD MANUFACTURING METHOD AND MOLD STRU...,Disclosed is a method of manufacturing a maste...,


In [25]:
# Conta il numero totale di famiglie uniche intercettate dalla query principale
total_families_query = sector_ids.count()

print(f"Numero totale di famiglie di brevetti trovate per il settore '{SECTOR_NAME}' (2010-2023): {total_families_query}")

Numero totale di famiglie di brevetti trovate per il settore 'Quantum Computing' (2010-2023): 33


In [29]:
#da mettere poi nel csv
y_true = [
    0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 
    1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 
    0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 
    0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 
    0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 
    0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 
    1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 
    0, 0, 0
]

In [31]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

# Calcolo delle metriche di classificazione
#y_true = df_validation["manual_label"].astype(int)
y_pred = df_validation["query_label"].astype(int)

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
conf_matrix = confusion_matrix(y_true, y_pred)

print(f"--- RISULTATI VALIDAZIONE ---")
print(f"Accuracy:  {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Precision: {precision:.4f} ({precision * 100:.2f}%)")
print(f"Recall:    {recall:.4f} ({recall * 100:.2f}%)")


--- RISULTATI VALIDAZIONE ---
Accuracy:  0.7952 (79.52%)
Precision: 0.4848 (48.48%)
Recall:    1.0000 (100.00%)
